In [1]:
import pandas as pd


url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [2]:
df["Survived"].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

### Pregunta A — Sumarización categórica

**Respuesta:** la tasa de supervivencia global fue del **38.38%**. Murió el **61.62%** de los 891 pasajeros.

In [3]:
df.groupby("Sex")["Survived"].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

### Pregunta B — Agrupación y agregación

**Respuesta:** sobrevivió el **74.20%** de las mujeres contra el **18.89%** de los hombres — una mujer tenía **3.9 veces** más probabilidad de sobrevivir. El código marítimo queda confirmado.

Funciona porque `Survived` está codificada 0/1: su promedio *es* la proporción de sobrevivientes del grupo.

In [4]:
cuartiles = df["Fare"].quantile([0.25, 0.75])
q1 = cuartiles[0.25]
q3 = cuartiles[0.75]
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

print("Q1:", q1, "| Q3:", q3)
print("IQR:", iqr)
print("Limite superior:", limite_superior)

outliers = df[df["Fare"] > limite_superior]

print("\nPasajeros por encima del limite:", len(outliers))
print("\nDistribucion por clase:")
print(outliers["Pclass"].value_counts())

Q1: 7.9104 | Q3: 31.0
IQR: 23.0896
Limite superior: 65.6344

Pasajeros por encima del limite: 116

Distribucion por clase:
Pclass
1    104
3      7
2      5
Name: count, dtype: int64


### Pregunta C — Confirmación de outliers con IQR

**Respuesta:**

| | |
|---|---|
| Q1 (25%) | 7.9104 |
| Q3 (75%) | 31.0000 |
| IQR = Q3 − Q1 | 23.0896 |
| Límite superior = Q3 + 1.5·IQR | 65.6344 |

**116 pasajeros** (13.02%) pagaron por encima del límite. La mayoría era de **primera clase: 104 de 116 (89.7%)**; el resto se reparte entre tercera (7) y segunda (5).

No hay indicio de fraude: los outliers se concentran justo en la clase que pagaba más.

In [5]:
print("Media:  ", df["Fare"].mean())
print("Mediana:", df["Fare"].median())
print("Asimetria (skew):", df["Fare"].skew())

Media:   32.204207968574636
Mediana: 14.4542
Asimetria (skew): 4.787316519674893


### Pregunta D — Media vs mediana y el efecto en KNN

**Respuesta:** media **32.20** contra mediana **14.45** (skew **+4.79**). Que la media supere tanto a la mediana significa *asimetría positiva*: existe una cola larga de tarifas altas que jala la media (promedio aritmético, sensible a extremos) sin mover la mediana (valor que parte la muestra en dos). La media deja de representar al pasajero típico.

**En KNN sin escalar:** la distancia euclidiana suma diferencias al cuadrado, así que domina la variable de mayor rango. `Fare` va de 0 a 512 y `Pclass` de 1 a 3; una diferencia de 100 libras pesa ~1,600 veces más que una de 2 clases (100² vs 2²). El modelo clasificaría prácticamente **solo con `Fare`**, ignorando el resto. Corrección: `np.log1p` para comprimir la cola y escalado robusto (mediana e IQR).

In [6]:
proporciones = df["Survived"].value_counts(normalize=True)

muestra = pd.concat([
    df[df["Survived"] == clase].sample(n=round(proporciones[clase] * 150), random_state=42)
    for clase in proporciones.index
])

print("Tamano de la muestra:", len(muestra))
print("\nProporcion en la muestra:")
print(muestra["Survived"].value_counts(normalize=True))
print("\nProporcion en la base completa:")
print(proporciones)

Tamano de la muestra: 150

Proporcion en la muestra:
Survived
0    0.613333
1    0.386667
Name: proportion, dtype: float64

Proporcion en la base completa:
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


### Pregunta E — Muestreo estratificado de 150 pasajeros

**Respuesta:** el código separa por clase, toma de cada estrato las filas que le corresponden según su proporción real y las une. Resultado: **150 exactos** — 92 no sobrevivientes (61.33%) y 58 sobrevivientes (38.67%), contra 61.62% / 38.38% en la base completa. La diferencia de 0.3 pp es puro redondeo a entero.

**Sesgo que se evita:** el **sesgo de muestreo** por desbalance de clases. Con `df.sample(150)` la proporción de sobrevivientes queda a merced del azar (podría salir 30% o 50%), y el modelo aprendería una *tasa base* equivocada, con probabilidades calibradas contra una realidad que no existe.

In [7]:
print(df.groupby("Survived")["Age"].mean())

print("\nDonde estan los 177 NaN de Age:")
print(df[df["Age"].isnull()].groupby(["Pclass", "Survived"]).size())

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

Donde estan los 177 NaN de Age:
Pclass  Survived
1       0            16
        1            14
2       0             7
        1             4
3       0           102
        1            34
dtype: int64


### Pregunta F — El sesgo de ignorar los NaN

**Respuesta:** el sesgo es de **selección**: los faltantes no son *MCAR* (*Missing Completely At Random*), así que descartarlos no descarta una muestra representativa.

De los 177 NaN, **136 son de tercera clase** y **102 son de tercera que además murieron**. El promedio de 30.63 años del grupo "murió" se calculó excluyendo a 125 personas, casi todas de tercera — el grupo más joven del barco (mediana 24 años contra 37 en primera).

**Consecuencia:** ese promedio queda inflado hacia el perfil de las clases altas, la brecha real de edad entre muertos y sobrevivientes se enmascara, y el modelo subestimaría el peso de la edad y la clase en la mortalidad.

### Pregunta G — ¿Eliminar los boletos de 512 libras con `.drop()`?

**Respuesta: no.** Son **señal, no ruido**, y se puede demostrar: las tres tarifas de 512.3292 son de `Pclass = 1`, comparten el mismo `Ticket` (PC 17755) y tienen camarotes registrados (B51-B53-B55 y B101). Es un boleto de grupo para una suite de lujo, no un error de captura — y las tres personas sobrevivieron.

Borrarlas sería contraproducente:

- **Elimina la señal más predictiva.** 104 de los 116 outliers son de primera clase, que sobrevivió al 62.96% contra el 24.24% de tercera. Filtrar por `Fare` equivale a borrar casi la mitad de la primera clase, justo la relación tarifa → clase → supervivencia que el modelo debe aprender.
- **El problema es la escala, no la existencia.** La solución es transformar (`np.log1p`) o escalar con `RobustScaler`, no perder registros.

Un outlier se elimina solo cuando es *físicamente imposible* (edad de 300 años, tarifa negativa). En duda, se conserva y se marca con una bandera.

In [8]:
df["Titulo"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

print("Mediana global de Age:", df["Age"].median())
print("\nMediana de Age por titulo:")
print(df.groupby("Titulo")["Age"].median().sort_values())

Mediana global de Age: 28.0

Mediana de Age por titulo:
Titulo
Master       3.5
Miss        21.0
Mlle        24.0
Mme         24.0
Ms          28.0
Mr          30.0
Countess    33.0
Mrs         35.0
Jonkheer    38.0
Don         40.0
Dr          46.5
Rev         46.5
Lady        48.0
Major       48.5
Sir         49.0
Col         58.0
Capt        70.0
Name: Age, dtype: float64


### Pregunta H — ¿Por qué imputar por título es superior a la mediana global?

**Respuesta:** porque la edad está fuertemente estructurada por el título, y la mediana global promedia esa estructura hasta borrarla.

| Título | Mediana de Age |
|---|---|
| `Master` (niño) | **3.5** |
| `Miss` | 21.0 |
| `Mr` | 30.0 |
| `Mrs` | 35.0 |
| *Global* | *28.0* |

Imputar 28 años a un `Master` produce un error de **~24 años**, y no es marginal: hay 40 `Master` en el dataset.

**Estadísticamente:** imputar una constante global reduce artificialmente la varianza y *atenúa* la correlación real entre `Age` y `Survived` (la diluye hacia cero). Imputar por grupo conserva la varianza entre grupos y estima condicionando a información que sí conocemos. Menor error de imputación, menor error del modelo — y `Name`, que parecía texto libre inservible, resulta el mejor predictor de edad disponible.

In [9]:
print("Varianza de Survived:", df["Survived"].var())

p = df["Survived"].mean()
print("Comprobacion p * (1 - p):", p * (1 - p))

Varianza de Survived: 0.2367722165474984
Comprobacion p * (1 - p): 0.23650647893072133


### Pregunta I — Varianza de `Survived`

**Respuesta:** **0.2368**. Para una binaria 0/1 la varianza es `p · (1 − p)`: `0.3838 × 0.6162 = 0.2365` (la diferencia mínima es porque pandas usa varianza muestral, con `n − 1`). Su máximo posible es 0.25, en `p = 0.5`.

**Si fuera 0.0:** todos los pasajeros tendrían el mismo valor — o todos murieron, o todos sobrevivieron. La columna sería una constante y su contenido informativo, **cero**.

**Al entrenar un clasificador:** imposible. Con una sola clase no hay frontera de decisión que aprender ni contraste del cual estimar probabilidades; `scikit-learn` lanza error directamente al exigir al menos dos clases en `y`. Si no lo hiciera, predeciría siempre esa clase con un 100% de exactitud aparente y cero capacidad de generalización. Lo mismo aplica a una variable *predictora* con varianza 0: no discrimina nada y debe eliminarse.

In [10]:
conteos = df.groupby(["Pclass", "Sex", "Embarked"])["PassengerId"].count()
print(conteos)

print("\nSubgrupos con 2 pasajeros o menos:", (conteos <= 2).sum(), "de", len(conteos))

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64

Subgrupos con 2 pasajeros o menos: 4 de 18


### Pregunta J — Grupos de 1 pasajero y la maldición de la dimensionalidad

**Respuesta: sobreajuste (*overfitting*).**

El cruce de tres variables fragmenta el dataset en 18 subgrupos, y **4 tienen 2 pasajeros o menos** (1ª/mujer/Queenstown = 1, 1ª/hombre/Queenstown = 1, 2ª/hombre/Queenstown = 1, 2ª/mujer/Queenstown = 2).

Un subgrupo de un solo pasajero produce una probabilidad de 0% o 100%: una regla de certeza aparente construida sobre una única observación. Si ese pasajero hubiera tenido la suerte contraria, la regla se invertiría por completo — es **ruido de muestreo**, no patrón. El modelo lo memoriza: error de entrenamiento bajísimo, error de generalización alto. Eso es exactamente sobreajuste. El subajuste es lo opuesto (modelo demasiado simple); aquí sobra especificidad, no falta.

**Mitigaciones:** exigir un mínimo de observaciones por grupo (`min_samples_leaf`), agrupar categorías de baja frecuencia en "Otros", o aplicar suavizado bayesiano a los grupos pequeños.